# Notebook 07: Inference Pipeline

End-to-end latency benchmark and structured output demo for HeartBERT LoRA wrapped in `HeartBERTPipeline`.

**Contents:**
1. Load PTB-XL test set (fold 10), sample 20 records (4 per superclass)
2. Load HeartBERT LoRA, wrap in `HeartBERTPipeline`
3. Benchmark end-to-end latency (tokenisation + inference, 5 warm-up + 50 timed runs)
4. Latency chart (mean / median / p95)
5. Demo: run 20 records, inspect 3 result dicts
6. Save `results/07_inference_pipeline/latency.json`

In [1]:
import os, sys, json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

sys.path.insert(0, os.path.abspath('..'))

from src.utils.seed import set_seed
set_seed(42)

SUPERCLASSES = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
LEAD_NAMES   = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
RESULTS_DIR  = '../results/07_inference_pipeline'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Setup complete.')

Setup complete.


## 1. Load Test Data

Fold 10 is the held-out test set. 4 records per superclass are sampled with seed 42 for reproducibility.

In [ ]:
import wfdb
from src.utils.config import CFG
from src.preprocessing.label_utils import load_all_labels

DATA_PATH = CFG['data']['path']

df      = load_all_labels(DATA_PATH + 'ptbxl_database.csv', DATA_PATH + 'scp_statements.csv')
test_df = df[df['strat_fold'] == 10].reset_index(drop=True)

np.random.seed(42)
sampled = []
for cls in SUPERCLASSES:
    idx    = SUPERCLASSES.index(cls)
    mask   = test_df['label_vec'].apply(lambda v: v[idx] == 1.0)
    chosen = test_df[mask].sample(n=min(4, int(mask.sum())), random_state=42)
    sampled.append(chosen)

sample_df   = pd.concat(sampled).drop_duplicates().reset_index(drop=True)
print(f'Sampled {len(sample_df)} records.')

def _load(row):
    sig, _ = wfdb.rdsamp(DATA_PATH + row['filename_lr'])
    return sig.astype(np.float32)   # (1000, 12)

signals_raw = [_load(row) for _, row in sample_df.iterrows()]
true_labels = [row['label_vec'] for _, row in sample_df.iterrows()]
print(f'Signals loaded: {len(signals_raw)} x (1000, 12)')

## 2. Load HeartBERT LoRA

Loads roberta-base (HeartBERT architecture), attaches the LoRA adapter trained in notebook 04, and wraps it in `HeartBERTPipeline`.

In [ ]:
from src.models.heartbert import HeartBERTClassifier
from src.inference.pipeline import HeartBERTPipeline

hb = HeartBERTClassifier(num_labels=5)
hb.load()
hb.apply_peft(r=8, alpha=16, use_dora=False)
hb.load_adapter('../results/heartbert_lora_r8/best_adapter')

pipeline = HeartBERTPipeline(hb)
print('HeartBERTPipeline ready.')

## 3. Latency Benchmark

5 warm-up passes (not measured) followed by 50 timed passes. Each pass covers the full path: Lead II extraction -> tokenisation (letter encoding + RoBERTa tokenizer) -> LoRA-adapted forward pass -> sigmoid.

In [ ]:
print('Benchmarking (5 warm-up + 50 timed runs)...')
report = pipeline.benchmark(signals_raw, n_warmup=5, n_runs=50)

print(f"\nDevice         : {report['device']}")
print(f"Runs           : {report['n_runs']}")
print(f"Mean latency   : {report['latency_ms_mean']:.1f} ms")
print(f"Median latency : {report['latency_ms_median']:.1f} ms")
print(f"P95 latency    : {report['latency_ms_p95']:.1f} ms")
print(f"Throughput     : {report['throughput_per_sec']:.0f} records/s")

## 4. Latency Chart

In [ ]:
bar_labels = ['Mean', 'Median', 'P95']
bar_values = [
    report['latency_ms_mean'],
    report['latency_ms_median'],
    report['latency_ms_p95'],
]

fig, ax = plt.subplots(figsize=(6, 3))
bars = ax.barh(bar_labels, bar_values, color=['#4C9BE8', '#2C7BB6', '#D7191C'])
ax.set_xlabel('Latency (ms)')
ax.set_title('HeartBERT LoRA -- End-to-End Latency per Record')
for bar, val in zip(bars, bar_values):
    ax.text(val + 0.3, bar.get_y() + bar.get_height() / 2,
            f'{val:.1f} ms', va='center', fontsize=10)
ax.set_xlim(0, max(bar_values) * 1.25)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/latency_chart.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved latency_chart.png')

## 5. Pipeline Demo

Run all sampled records through the pipeline and inspect 3 representative result dicts.

In [ ]:
all_results = [pipeline.predict(sig) for sig in signals_raw]
print(f'Ran {len(all_results)} records through the pipeline.\n')

print('--- 3 sample results ---\n')
for i in [0, 5, 10]:
    true_cls = [SUPERCLASSES[j] for j, v in enumerate(true_labels[i]) if v == 1.0]
    print(f'Record {i:2d}  |  True label: {true_cls}')
    print(json.dumps(all_results[i], indent=2))
    print()

## 6. Save Results

In [ ]:
output = {
    'benchmark':      report,
    'n_records':      len(all_results),
    'sample_results': all_results[:3],
}
out_path = f'{RESULTS_DIR}/latency.json'
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)
print(f'Saved {out_path}')